In [81]:
import spark
#imports
from pyspark.sql.functions import *
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, StringType, TimestampType, BooleanType
import ConnectionConfig as cc

In [72]:
#config
cc.setupEnvironment()
print(cc.config.sections())

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [50]:
#Cluster aanmaken
spark = cc.startLocalCluster("DIM_USER",4)
spark.getActiveSession()

In [73]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [80]:
#get info
user_src = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option(
        "dbtable",
        "(select id, first_name, last_name, mail as email, street || ' ' || number as address from user_table) as subq"
    ) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()\


In [82]:
user_src.createOrReplaceTempView("dimUserTemp")

df_user_dim = spark.sql("""
SELECT
  id,
  first_name,
  last_name,
  email,
  street,
    CASE WHEN t.treasure_count > 1
             THEN True ELSE False END as dedicator,
  to_timestamp('1999-01-01','yyyy-MM-dd') as scd_start,
  to_timestamp('2100-12-12','yyyy-MM-dd') as scd_end,
  True as current
FROM dimUserTemp
LEFT JOIN (
        SELECT user_id, COUNT(*) as treasure_count
        FROM treasure
        GROUP BY owner_id
    ) t
    ON d.id = t.owner_id""")

In [84]:
#opslagen tabel
user_src.write.format("delta").mode("overwrite").save("delta/USER_DIM")

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 83c40d95-b1a7-44a0-9bc5-a4f6dddc6db6).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- id: binary (nullable = true)
-- first_name: string (nullable = true)
-- last_name: string (nullable = true)
-- email: string (nullable = true)
-- address: string (nullable = true)


Data schema:
root
-- id: binary (nullable = true)
-- first_name: string (nullable = true)
-- last_name: string (nullable = true)
-- email: string (nullable = true)
-- address: string (nullable = true)
-- country: string (nullable = true)
-- experienceLevel: string (nullable = true)
-- dedicator: boolean (nullable = true)
-- start_scd: timestamp 

In [47]:
spark.stop()